# Colour and power Doppler with a linear array (GUI4us in a notebook)

The same environment and displays as `run.py` next to this notebook: two TX/RX sequences -- the B-mode
(synthetic transmit aperture, coherent/incoherent compounding, SRAD speckle filter) and the Doppler ensemble
(32 plane waves at 10 deg) -- processed by a Graph with one pipeline per sequence; colour and power Doppler
(wall filter, persistence) overlaid on the B-mode (the Doppler estimates outside their dynamic range are NaN,
i.e. transparent).

Requirements: ARRUS (e.g. the release wheel for your platform), cupy, scipy and
`pip install "gui4us[jupyter]"`. The us4R session belongs to this kernel: run the last cell (or
restart the kernel) to release it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from gui4us.model import StreamDataId
from run import (BMODE_DRANGE, BMODE_OUTPUT, COLOR_OUTPUT, COLOR_RANGE, POWER_DRANGE,
                 POWER_OUTPUT, create_gui)   # run.py in this directory

In [ ]:
SESSION_CFG = os.environ.get("ARRUS_SESSION_CFG", "us4r.prototxt")
VOLTAGE = 10

In [ ]:
gui = create_gui(SESSION_CFG, voltage=VOLTAGE, view="none")
gui.start()

## Live view

Colour and power Doppler, one above the other; the Doppler thresholds, the persistence and (with
`--clutter-filter svd` / `create_gui(..., clutter_filter="svd")`) the SVD wall filter are in the control panel.

In [ ]:
gui.widget()

## Data into variables

In [ ]:
gui.capture(5, timeout=60)
bmode = gui.captured_array(BMODE_OUTPUT)   # [dB]
color = gui.captured_array(COLOR_OUTPUT)   # velocity; NaN: no flow
power = gui.captured_array(POWER_OUTPUT)   # [dB]; NaN: no flow
print(bmode.shape, color.shape, power.shape)
print(f"pixels with flow in the last frame: {100*np.isfinite(color[-1]).mean():.1f} %")

In [ ]:
# Overlays, as in the displays: NaN pixels of the Doppler images are transparent.
(z_min, z_max), (x_min, x_max) = gui.metadata.output(StreamDataId("default", BMODE_OUTPUT)).extents
extent = np.array([x_min, x_max, z_max, z_min])*1e3
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, overlay, cmap, value_range, title in (
        (axes[0], color[-1], "bwr", COLOR_RANGE, "Color Doppler"),
        (axes[1], power[-1], "hot", POWER_DRANGE, "Power Doppler")):
    ax.imshow(bmode[-1], cmap="gray", vmin=BMODE_DRANGE[0], vmax=BMODE_DRANGE[1], extent=extent)
    ax.imshow(overlay, cmap=cmap, vmin=value_range[0], vmax=value_range[1], extent=extent)
    ax.set_title(title); ax.set_xlabel("OX [mm]"); ax.set_ylabel("OZ [mm]")

In [ ]:
# The Doppler thresholds, from code (the same settings as the control panel).
print([setting.name for setting in gui.settings])

In [ ]:
gui.close()